In [1]:
!pip install chromadb sentence-transformers pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [2]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

In [3]:
from google.colab import files
uploaded = files.upload()

Saving bbc-text.csv.zip to bbc-text.csv (1).zip


In [5]:
import zipfile

# Assuming the uploaded file is 'bbc-text.csv (1).zip'
zip_file_name = 'bbc-text.csv (1).zip'
extracted_file_name = 'bbc-text.csv' # The name of the csv file inside the zip

# Unzip the file
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall()

# Now read the extracted CSV file
df = pd.read_csv(extracted_file_name)
df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [6]:
print(df.columns)
print(df.shape)

Index(['category', 'text'], dtype='object')
(2225, 2)


In [7]:
df = df[['category', 'text']].dropna()
df = df.head(100)   # you can increase later
df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [8]:
client = chromadb.Client()

collection = client.create_collection(name="rag_collection")

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
documents = df['text'].tolist()
ids = [f"doc{i+1}" for i in range(len(documents))]

print("Total Documents:", len(documents))
print("Sample ID:", ids[:5])

Total Documents: 100
Sample ID: ['doc1', 'doc2', 'doc3', 'doc4', 'doc5']


In [11]:
embeddings = model.encode(documents)
print("Embedding shape:", len(embeddings), len(embeddings[0]))

Embedding shape: 100 384


In [12]:
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=ids
)

In [13]:
query = "What is artificial intelligence?"

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)

print(results)

{'ids': [['doc95', 'doc81']], 'embeddings': None, 'documents': [['amnesty chief laments war failure the lack of public outrage about the war on terror is a powerful indictment of the failure of human rights groups  amnesty international s chief has said.  in a lecture at the london school of economics  irene khan said human rights had been flouted in the name of security since 11 september  2001. she said the human rights movement had to use simpler language both to prevent scepticism and spread a moral message. and it had to fight poverty  not just focus on political rights for elites.  ms khan highlighted detentions without trial  including those at the us camp at guantanamo bay in cuba  and the abuse of prisoners as evidence of increasing human rights problems.  what s a new challenge is the way in which this age-old debate on security and human rights has been translated into the language of war   she said.  by using the language of war  human rights are being sidelined because we 

In [14]:
for doc in results['documents'][0]:
    print("\nRetrieved Document:\n")
    print(doc)
    print("-" * 80)


Retrieved Document:

amnesty chief laments war failure the lack of public outrage about the war on terror is a powerful indictment of the failure of human rights groups  amnesty international s chief has said.  in a lecture at the london school of economics  irene khan said human rights had been flouted in the name of security since 11 september  2001. she said the human rights movement had to use simpler language both to prevent scepticism and spread a moral message. and it had to fight poverty  not just focus on political rights for elites.  ms khan highlighted detentions without trial  including those at the us camp at guantanamo bay in cuba  and the abuse of prisoners as evidence of increasing human rights problems.  what s a new challenge is the way in which this age-old debate on security and human rights has been translated into the language of war   she said.  by using the language of war  human rights are being sidelined because we know human rights do not apply in times of w

In [15]:
def retrieve(query):
    query_embedding = model.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=2
    )
    return results['documents'][0]

In [16]:
print(retrieve("Explain machine learning"))

['microsoft gets the blogging bug software giant microsoft is taking the plunge into the world of blogging.  it is launching a test service to allow people to publish blogs  or online journals  called msn spaces. microsoft is trailing behind competitors like google and aol  which already offer services which make it easy for people to set up web journals. blogs  short for web logs  have become a popular way for people to talk about their lives and express opinions online.  msn spaces is free to anyone with a hotmail or msn messenger account. people will be able to choose a layout for the page  upload images and share photo albums and music playlists. the service will be supported by banner ads.  this is a simple tool for people to express themselves   said msn s blake irving. this is microsoft s first foray into blogging  which has taken off as a web phenomenon in the past year. competitors like google already offer free services through its blogger site  while aol provides its members

In [17]:
print("\nQuery 1 Results:")
print(retrieve("Tell me about technology"))

print("\nQuery 2 Results:")
print(retrieve("What is politics?"))

print("\nQuery 3 Results:")
print(retrieve("Sports news"))


Query 1 Results:
['pandas benefit from wireless net the world s dwindling panda population is getting a helping hand from a wireless internet network.  the wolong nature reserve in the sichuan province of southwest china is home to 20% of the remaining 1 500 giant pandas in the world. a broadband and wireless network installed on the reserve has allowed staff to chronicle the pandas  daily activities. the data and images can be shared with colleagues around the world. the reserve conducts vital research on both panda breeding and bamboo ecology.  using the network  vets have been able to observe how infant pandas feed and suggest changes to improve the tiny cubs  chances of survival.   digital technology has transformed the way we communicate and share information inside wolong and with the rest of the world   said zhang hemin  director of the wolong nature reserve.  our researchers now have state-of-the-art digital technology to help foster the panda population and manage our preciou

In [18]:
metadatas = [{"category": cat} for cat in df['category'].tolist()]

In [19]:
client = chromadb.Client()
collection = client.create_collection(name="rag_collection_with_metadata")

In [20]:
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=ids,
    metadatas=metadatas
)

In [21]:
query = "latest technology ideas"
query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)

print(results)

{'ids': [['doc25', 'doc1']], 'embeddings': None, 'documents': [['mobile audio enters new dimension as mobile phones move closer to being a ubiquitous  all-in-one media player  audio is becoming ever more important. but how good can that sound be from such a small device   the sound of a buzzing bee jumps from left to right before disappearing around the back of my head. the surround sound demo is unremarkable when heard on a multi-speaker home cinema system but startling when emerging from a small mobile phone. british firm sonaptic is one of a number of companies to have developed 3d audio technology that emerges from stereo speakers. firms am3d and srs both offer stereo-widening technology for mobile phones. but sonaptic s managing director david monteith says his firm is the only company to offer positional 3d audio on a mobile.   there are quite a few basic technologies out there  making the sound seem a bit bigger  headphones a bit nicer.   no-one has really tried before to make p

In [22]:
for i in range(len(results['documents'][0])):
    print(f"\nResult {i+1}")
    print("Category:", results['metadatas'][0][i]['category'])
    print("Text:", results['documents'][0][i][:500])   # first 500 characters
    print("-" * 80)


Result 1
Category: tech
Text: mobile audio enters new dimension as mobile phones move closer to being a ubiquitous  all-in-one media player  audio is becoming ever more important. but how good can that sound be from such a small device   the sound of a buzzing bee jumps from left to right before disappearing around the back of my head. the surround sound demo is unremarkable when heard on a multi-speaker home cinema system but startling when emerging from a small mobile phone. british firm sonaptic is one of a number of comp
--------------------------------------------------------------------------------

Result 2
Category: tech
Text: tv future in the hands of viewers with home theatre systems  plasma high-definition tvs  and digital video recorders moving into the living room  the way people watch tv will be radically different in five years  time.  that is according to an expert panel which gathered at the annual consumer electronics show in las vegas to discuss how these new techno

In [23]:
def retrieve(query, n_results=2):
    query_embedding = model.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    for i in range(len(results['documents'][0])):
        print(f"\nResult {i+1}")
        print("Category:", results['metadatas'][0][i]['category'])
        print("Text:", results['documents'][0][i][:500])
        print("-" * 80)

In [24]:
retrieve("machine learning and technology")


Result 1
Category: tech
Text: pandas benefit from wireless net the world s dwindling panda population is getting a helping hand from a wireless internet network.  the wolong nature reserve in the sichuan province of southwest china is home to 20% of the remaining 1 500 giant pandas in the world. a broadband and wireless network installed on the reserve has allowed staff to chronicle the pandas  daily activities. the data and images can be shared with colleagues around the world. the reserve conducts vital research on both pa
--------------------------------------------------------------------------------

Result 2
Category: tech
Text: tv future in the hands of viewers with home theatre systems  plasma high-definition tvs  and digital video recorders moving into the living room  the way people watch tv will be radically different in five years  time.  that is according to an expert panel which gathered at the annual consumer electronics show in las vegas to discuss how these new techno

In [25]:
df_ai = df[df['category'] == 'tech'].copy()
df_ai.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
19,tech,games maker fights for survival one of britain...
20,tech,security warning over fbi virus the us feder...
21,tech,halo 2 heralds traffic explosion the growing p...
24,tech,mobile audio enters new dimension as mobile ph...


In [26]:
df.to_csv("processed_rag_dataset.csv", index=False)
print("Saved successfully!")

Saved successfully!
